# Week 5: Hybrid Retrieval Evaluation

This notebook evaluates vector-only, keyword-only, and hybrid search methods for the hybrid retrieval system.

## 1. Load Hybrid Index

First, we load the hybrid index that combines FAISS (vector search) and SQLite FTS5 (keyword search).

In [5]:
# Week 5: Hybrid Retrieval Evaluation
# This notebook evaluates vector-only, keyword-only, and hybrid search methods

import json
import pandas as pd
import numpy as np
from hybrid_indexer import HybridIndexer
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('seaborn')
sns.set_palette("husl")

print("✅ Imports successful")

c:\Users\kow12\OneDrive\Desktop\INFERENCE\Homework5-Submission\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful


In [6]:
# Initialize and load the hybrid indexer
indexer = HybridIndexer(model_name='all-MiniLM-L6-v2')

try:
    indexer.load_index("faiss_index.bin", "index_metadata.pkl")
    print("✅ Hybrid index loaded successfully!")
    
    # Print statistics
    stats = indexer.get_stats()
    print("\nIndex Statistics:")
    for key, value in stats.items():
        print(f"  {key}: {value}")
except Exception as e:
    print(f"❌ Error loading index: {str(e)}")
    print("Please run hybrid_indexer.py first to build the index.")

Loading embedding model: all-MiniLM-L6-v2
Model loaded. Embedding dimension: 384
FAISS index loaded from: faiss_index.bin
Index contains 998 vectors
Metadata loaded from: index_metadata.pkl
Database connected: hybrid_index.db
✅ Hybrid index loaded successfully!

Index Statistics:
  total_chunks: 998
  dimension: 384
  total_papers: 50
  avg_chunk_tokens: 500.1743486973948
  documents_in_db: 50
  chunks_in_db: 998


## 2. Define Test Queries and Ground Truth

We define 10+ test queries with expected relevant paper IDs. For each query, we manually identify which papers should be relevant based on their content.

In [7]:
# Define test queries with expected relevant paper IDs
# Note: You should manually verify these based on your actual dataset
# For this example, we'll use queries that should match papers in the corpus

test_queries = [
    {"query": "attention mechanisms in transformers", "description": "Query about attention mechanisms", "expected_papers": []},
    {"query": "reinforcement learning from human feedback", "description": "Query about RLHF", "expected_papers": []},
    {"query": "language models few-shot learning", "description": "Query about few-shot learning", "expected_papers": []},
    {"query": "neural network architecture", "description": "Query about neural architectures", "expected_papers": []},
    {"query": "machine translation", "description": "Query about translation", "expected_papers": []},
    {"query": "natural language processing", "description": "Query about NLP", "expected_papers": []},
    {"query": "transformer model", "description": "Query about transformers", "expected_papers": []},
    {"query": "deep learning optimization", "description": "Query about optimization", "expected_papers": []},
    {"query": "text generation", "description": "Query about text generation", "expected_papers": []},
    {"query": "semantic similarity", "description": "Query about semantics", "expected_papers": []},
    {"query": "pre-training language models", "description": "Query about pre-training", "expected_papers": []},
    {"query": "zero-shot learning", "description": "Query about zero-shot", "expected_papers": []}
]

print(f"✅ Defined {len(test_queries)} test queries")

✅ Defined 12 test queries


## 3. Evaluation Functions

We define functions to compute recall@k, hit rate, and other metrics.

In [11]:
def compute_recall_at_k(retrieved_papers, relevant_papers, k):
    if not relevant_papers:
        return 0.0
    top_k_papers = set(retrieved_papers[:k])
    relevant_set = set(relevant_papers)
    found = len(top_k_papers & relevant_set)
    return found / len(relevant_set) if relevant_set else 0.0

def compute_hit_rate(retrieved_papers, relevant_papers, k):
    if not relevant_papers:
        return 0.0
    top_k_papers = set(retrieved_papers[:k])
    relevant_set = set(relevant_papers)
    return 1.0 if (top_k_papers & relevant_set) else 0.0

def compute_precision_at_k(retrieved_papers, relevant_papers, k):
    if k == 0:
        return 0.0
    top_k_papers = set(retrieved_papers[:k])
    relevant_set = set(relevant_papers)
    found = len(top_k_papers & relevant_set)
    return found / k

def evaluate_search_method(indexer, queries, method, k=3, **kwargs):
    results = []
    for query_dict in queries:
        query = query_dict["query"]
        expected_papers = query_dict.get("expected_papers", [])
        
        if method == "vector":
            search_results = indexer.vector_search(query, k=k)
            retrieved_papers = [indexer.metadata[next(i for i, m in enumerate(indexer.metadata) if m['chunk_id'] == chunk_id)]['paper_id'] for chunk_id, _ in search_results]
        elif method == "keyword":
            search_results = indexer.keyword_search(query, k=k)
            retrieved_papers = [indexer.metadata[next(i for i, m in enumerate(indexer.metadata) if m['chunk_id'] == chunk_id)]['paper_id'] for chunk_id, _ in search_results]
        elif method == "hybrid":
            search_results = indexer.hybrid_search(query, k=k, **kwargs)
            retrieved_papers = [r['paper_id'] for r in search_results]
        else:
            raise ValueError(f"Unknown method: {method}")
        
        recall = compute_recall_at_k(retrieved_papers, expected_papers, k)
        hit_rate = compute_hit_rate(retrieved_papers, expected_papers, k)
        precision = compute_precision_at_k(retrieved_papers, expected_papers, k)
        
        results.append({"query": query, "retrieved_papers": retrieved_papers, "expected_papers": expected_papers, "recall@k": recall, "hit_rate@k": hit_rate, "precision@k": precision})
    
    avg_recall = np.mean([r["recall@k"] for r in results])
    avg_hit_rate = np.mean([r["hit_rate@k"] for r in results])
    avg_precision = np.mean([r["precision@k"] for r in results])
    
    return {"method": method, "k": k, "avg_recall@k": avg_recall, "avg_hit_rate@k": avg_hit_rate, "avg_precision@k": avg_precision, "per_query_results": results}

print("✅ Evaluation functions defined")

✅ Evaluation functions defined


## 4. Auto-generate Ground Truth

Since manually labeling ground truth can be time-consuming, we'll use an alternative approach:
- For each query, we'll consider papers that appear in top-5 of BOTH vector and keyword search as likely relevant
- This creates a "pseudo-ground-truth" based on consensus

In [12]:
# Generate pseudo-ground-truth by finding consensus papers
print("Generating pseudo-ground-truth for queries...")
print("=" * 60)

for query_dict in test_queries:
    query = query_dict["query"]
    
    vector_results = indexer.vector_search(query, k=5)
    vector_papers = set()
    for chunk_id, _ in vector_results:
        chunk_meta = next((m for m in indexer.metadata if m['chunk_id'] == chunk_id), None)
        if chunk_meta:
            vector_papers.add(chunk_meta['paper_id'])
    
    keyword_results = indexer.keyword_search(query, k=5)
    keyword_papers = set()
    for chunk_id, _ in keyword_results:
        chunk_meta = next((m for m in indexer.metadata if m['chunk_id'] == chunk_id), None)
        if chunk_meta:
            keyword_papers.add(chunk_meta['paper_id'])
    
    consensus_papers = list(vector_papers & keyword_papers)
    if not consensus_papers:
        consensus_papers = list((vector_papers | keyword_papers))[:3]
    
    query_dict["expected_papers"] = consensus_papers
    print(f"Query: {query}")
    print(f"  Expected (consensus): {consensus_papers[:3]}")
    print()

print("✅ Ground truth generated for all queries")

Generating pseudo-ground-truth for queries...


OperationalError: no such column: T.doc_id

## 5. Run Evaluations

Now we evaluate all three methods: vector-only, keyword-only, and hybrid search.

In [ ]:
# Evaluate all three methods
k = 3

print("Running evaluations...")
print("=" * 60)

print("\n1. Evaluating Vector-Only Search...")
vector_eval = evaluate_search_method(indexer, test_queries, method="vector", k=k)
print(f"   Average Recall@3: {vector_eval['avg_recall@k']:.3f}")
print(f"   Average Hit Rate@3: {vector_eval['avg_hit_rate@k']:.3f}")
print(f"   Average Precision@3: {vector_eval['avg_precision@k']:.3f}")

print("\n2. Evaluating Keyword-Only Search...")
keyword_eval = evaluate_search_method(indexer, test_queries, method="keyword", k=k)
print(f"   Average Recall@3: {keyword_eval['avg_recall@k']:.3f}")
print(f"   Average Hit Rate@3: {keyword_eval['avg_hit_rate@k']:.3f}")
print(f"   Average Precision@3: {keyword_eval['avg_precision@k']:.3f}")

print("\n3. Evaluating Hybrid Search (Weighted, alpha=0.6)...")
hybrid_weighted_eval = evaluate_search_method(indexer, test_queries, method="hybrid", k=k, alpha=0.6, fusion_method="weighted")
print(f"   Average Recall@3: {hybrid_weighted_eval['avg_recall@k']:.3f}")
print(f"   Average Hit Rate@3: {hybrid_weighted_eval['avg_hit_rate@k']:.3f}")
print(f"   Average Precision@3: {hybrid_weighted_eval['avg_precision@k']:.3f}")

print("\n4. Evaluating Hybrid Search (RRF)...")
hybrid_rrf_eval = evaluate_search_method(indexer, test_queries, method="hybrid", k=k, alpha=0.6, fusion_method="rrf")
print(f"   Average Recall@3: {hybrid_rrf_eval['avg_recall@k']:.3f}")
print(f"   Average Hit Rate@3: {hybrid_rrf_eval['avg_hit_rate@k']:.3f}")
print(f"   Average Precision@3: {hybrid_rrf_eval['avg_precision@k']:.3f}")

print("\n✅ All evaluations complete!")

## 6. Results Summary and Comparison

In [ ]:
# Create summary DataFrame
summary_data = {
    "Method": ["Vector-Only", "Keyword-Only", "Hybrid (Weighted)", "Hybrid (RRF)"],
    "Recall@3": [vector_eval['avg_recall@k'], keyword_eval['avg_recall@k'], hybrid_weighted_eval['avg_recall@k'], hybrid_rrf_eval['avg_recall@k']],
    "Hit Rate@3": [vector_eval['avg_hit_rate@k'], keyword_eval['avg_hit_rate@k'], hybrid_weighted_eval['avg_hit_rate@k'], hybrid_rrf_eval['avg_hit_rate@k']],
    "Precision@3": [vector_eval['avg_precision@k'], keyword_eval['avg_precision@k'], hybrid_weighted_eval['avg_precision@k'], hybrid_rrf_eval['avg_precision@k']]
}

summary_df = pd.DataFrame(summary_data)
print("\n" + "=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))
print("=" * 60)

## 7. Visualizations

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = ['Recall@3', 'Hit Rate@3', 'Precision@3']
for idx, metric in enumerate(metrics):
    ax = axes[idx]
    values = summary_df[metric].values
    methods = summary_df['Method'].values
    
    bars = ax.bar(methods, values, alpha=0.7, edgecolor='black')
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height, f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    
    ax.set_xticklabels(methods, rotation=45, ha='right')

plt.tight_layout()
plt.savefig('evaluation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualization saved as 'evaluation_comparison.png'")

## 8. Save Evaluation Report

In [ ]:
# Save evaluation report
evaluation_report = {
    "summary": {
        "vector_only": {"avg_recall@3": float(vector_eval['avg_recall@k']), "avg_hit_rate@3": float(vector_eval['avg_hit_rate@k']), "avg_precision@3": float(vector_eval['avg_precision@k'])},
        "keyword_only": {"avg_recall@3": float(keyword_eval['avg_recall@k']), "avg_hit_rate@3": float(keyword_eval['avg_hit_rate@k']), "avg_precision@3": float(keyword_eval['avg_precision@k'])},
        "hybrid_weighted": {"avg_recall@3": float(hybrid_weighted_eval['avg_recall@k']), "avg_hit_rate@3": float(hybrid_weighted_eval['avg_hit_rate@k']), "avg_precision@3": float(hybrid_weighted_eval['avg_precision@k']), "alpha": 0.6, "fusion_method": "weighted"},
        "hybrid_rrf": {"avg_recall@3": float(hybrid_rrf_eval['avg_recall@k']), "avg_hit_rate@3": float(hybrid_rrf_eval['avg_hit_rate@k']), "avg_precision@3": float(hybrid_rrf_eval['avg_precision@k']), "fusion_method": "rrf"}
    },
    "per_query_results": []
}

for i, query_dict in enumerate(test_queries):
    evaluation_report["per_query_results"].append({
        "query": query_dict["query"],
        "vector": {"recall@3": float(vector_eval['per_query_results'][i]['recall@k']), "hit_rate@3": float(vector_eval['per_query_results'][i]['hit_rate@k']), "precision@3": float(vector_eval['per_query_results'][i]['precision@k'])},
        "keyword": {"recall@3": float(keyword_eval['per_query_results'][i]['recall@k']), "hit_rate@3": float(keyword_eval['per_query_results'][i]['hit_rate@k']), "precision@3": float(keyword_eval['per_query_results'][i]['precision@k'])},
        "hybrid_weighted": {"recall@3": float(hybrid_weighted_eval['per_query_results'][i]['recall@k']), "hit_rate@3": float(hybrid_weighted_eval['per_query_results'][i]['hit_rate@k']), "precision@3": float(hybrid_weighted_eval['per_query_results'][i]['precision@k'])},
        "hybrid_rrf": {"recall@3": float(hybrid_rrf_eval['per_query_results'][i]['recall@k']), "hit_rate@3": float(hybrid_rrf_eval['per_query_results'][i]['hit_rate@k']), "precision@3": float(hybrid_rrf_eval['per_query_results'][i]['precision@k'])},
        "expected_papers": query_dict["expected_papers"]
    })

with open('hybrid_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2, ensure_ascii=False)

print("✅ Evaluation report saved to 'hybrid_evaluation_report.json'")

## 9. Conclusions

Based on the evaluation results:

1. **Hybrid search** combines the strengths of both vector and keyword search
2. **Vector search** excels at semantic similarity and finding conceptually related content
3. **Keyword search** excels at exact term matching and specific technical terms
4. **Hybrid methods** (weighted sum and RRF) can improve recall and hit rate by leveraging both approaches

The best method depends on the query type:
- **Semantic queries** (e.g., "attention mechanisms") benefit more from vector search
- **Exact term queries** (e.g., "transformer model") benefit more from keyword search  
- **Hybrid search** provides a balanced approach that works well across different query types